# Reading groups of posts out of the semantic tree

`link_classification.ipynb` asks which **links** of a page are posts. This
notebook asks the question the selection screen actually asks: which **groups**
of the semantic tree are lists of posts, and which link of a group member is
the post.

The two questions have different answers. A group with twelve extra links
inside it costs the user one tick; a list broken into twenty-four groups of one
post costs the user twenty-four ticks and is unusable. So everything here is
counted per group as well as per link.

The tree is read from `../server/feature/feed/testdata/semantic/`, written by
`TestSemanticTree`, and the fixtures beside it say which URLs are posts. The
code shared with the other notebooks is in `tree.py`.

In [1]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, ".")
import tree

pages = tree.load()
print(f"{len(pages)} pages, "
      f"{sum(len(p.posts) for p in pages)} posts, "
      f"{sum(len(p.groups()) for p in pages)} candidate groups")

21 pages, 1070 posts, 720 candidate groups


In [2]:
# Reading the features of a group walks its whole subtree, and the searches
# further down read the same groups thousands of times.
_seen = {}


def features(members):
    key = tuple(id(m) for m in members)
    if key not in _seen:
        _seen[key] = tree.group_features(members)
    return _seen[key]

## What a group is, and what the best possible result would be

A **group** is a node of the tree with two or more children. Its members are
those children, and one member stands for one post.

Before looking for a rule, it is worth knowing what the tree allows at best.
The table below takes the fixtures in hand, picks the group that covers the
most posts, then the next one, and so on, until every post of the page is
covered. `ticks` is how many groups the user would have to select.

In [3]:
def oracle(page):
    rows = [({tree.normalize(l) for l in tree.group_links(g.children)
              if page.is_post(l)}, g) for g in page.groups()]
    need = set(page.posts)
    ticks = members = covered = 0
    while need:
        best = max(rows, key=lambda r: len(r[0] & need), default=None)
        if not best or not best[0] & need:
            break
        need -= best[0]
        ticks += 1
        members += len(best[1].children)
        covered += len(best[0])
    return {"posts": len(page.posts), "groups": len(page.groups()),
            "ticks": ticks, "uncovered": len(need),
            "selected members": members,
            "purity": round(covered / members, 2) if members else 0.0}


best_case = pd.DataFrame(
    [dict(page=p.name, **oracle(p)) for p in pages]
).set_index("page")
best_case

,posts,groups,ticks,uncovered,selected members,purity
page,,,,,,
anthropic.com-news,13,5,3,0,15,1.00
aws.amazon.com-jp-blogs-news,10,41,1,0,11,0.91
bbc.com,103,55,24,0,119,0.97
blog.google,8,6,2,0,8,1.00
claude.com-blog,24,10,3,0,28,0.96
cursor.com-blog,28,31,5,0,28,1.00
daily.bandcamp.com-album-of-the-day,30,13,1,0,30,1.00
daily.bandcamp.com-features,30,14,1,0,30,1.00
deepmind.google-blog,25,6,3,0,26,1.00


Every post is reachable, so the folding loses nothing. Three pages need many
ticks: `bbc.com` is a homepage of separate sections, and `github.blog` and
`qiita.com` are lists whose cards are still split into several nodes.

The number to beat is therefore about one or two groups per page for a blog
index, and more for a homepage where the sections are genuinely separate.

## Three problems, found by looking at the pages that fail

### 1. A card holds more links than the post

This is one card of `qiita.com`. The post link is there twice, next to the
author page, an event banner and five tag links.

In [4]:
qiita = {p.name: p for p in pages}["qiita.com"]


def dump(node, page, depth=0):
    mark = "post" if page.is_post(node.link) else "    "
    text = [t["value"][:38] for t in node.texts]
    print(f"{'  ' * depth}{mark} {node.link[-46:]:46s} {text}")
    for c in node.children:
        dump(c, page, depth + 1)


dump(qiita.root.children[1].children[2].children[6], qiita)

                                                    []
       qiita.com/official-events/dc6e42e0897543216e34 ['あなたはもう試した？判断特化AI「Jev」で遊ぼう！']
  post /qiita.com/shinkai_/items/61994be44d61c76716d3 []
       https://qiita.com/shinkai_                     ['@ shinkai_', '( 新海 正明 )', '2026-09-18']
  post /qiita.com/shinkai_/items/61994be44d61c76716d3 ['「LLMじゃないAI」が来た! TypeSafe AI「Jev」はなぜ文字列']
                                                      ['19']
                                                        []
           https://qiita.com/tags/%e7%94%9f%e6%88%90ai    ['生成AI']
           https://qiita.com/tags/typesafe                ['TypeSafe']
           https://qiita.com/tags/jev                     ['Jev']
           https://qiita.com/tags/ai                      ['AI']
           https://qiita.com/tags/llm                     ['LLM']


Three facts tell the extra links apart, and none of them reads the text of the
link:

- `qiita.com/shinkai_` is the beginning of the path of the post, so it is the
  page **about** the author rather than a post;
- the five `/tags/<name>` links share one shape and that shape occurs five
  times **inside one card**, so none of them can be the single post of the
  card;
- the event banner is carried by several cards of the same list, so it belongs
  to the list rather than to one card.

`tree.candidate_links` applies the three tests. The effect is measured below on
the group of each page that covers the most posts.

In [5]:
def target_group(page):
    """The group a perfect rule would pick: the one covering the most posts."""
    return max(page.groups(),
               key=lambda g: len({tree.normalize(l)
                                  for l in tree.group_links(g.children)
                                  if page.is_post(l)}))


rows = []
for p in pages:
    g = target_group(p)
    raw = [len({n.link for n in m.walk() if n.link}) for m in g.children]
    kept = [len(tree.candidate_links(m, g.children)) for m in g.children]
    rows.append({
        "page": p.name,
        "members": len(g.children),
        "links per member": round(sum(raw) / len(raw), 1),
        "candidates per member": round(sum(kept) / len(kept), 1),
        "members with one candidate":
            round(sum(1 for c in kept if c == 1) / len(kept), 2),
    })
pd.DataFrame(rows).set_index("page")

,members,links per member,candidates per member,members with one candidate
page,,,,
anthropic.com-news,10,1.0,1.0,1.00
aws.amazon.com-jp-blogs-news,11,6.1,1.4,0.82
bbc.com,15,8.7,7.1,0.33
blog.google,5,1.0,1.0,1.00
claude.com-blog,15,1.0,1.0,1.00
cursor.com-blog,12,1.0,1.0,1.00
daily.bandcamp.com-album-of-the-day,30,1.0,1.0,1.00
daily.bandcamp.com-features,30,1.0,1.0,1.00
deepmind.google-blog,12,1.0,1.0,1.00


This matters because "every member holds exactly one candidate link" is the
strongest single sign of a list of posts. Without the three tests the real
lists score badly on it, and small groups of navigation links score perfectly.

### 2. Choosing the link of a member needs the whole group

Three ways of choosing the link a member stands for, measured on the group of
each page that covers the most posts, 910 members in total:

- the **first link** in document order is right 887 times. It fails where the
  card puts something else first: 3 of 15 on `bbc.com`, 21 of 30 on `qiita.com`.
- ranking the candidates by how often the card repeats them and by the length
  of the text on them, **one pass**, is right 871 times. It repairs `bbc.com`
  and `qiita.com` and breaks `developer.apple.com`, where an item is running
  text and a link in its body is repeated more often than the item's own link:
  108 right becomes 74.
- reading the group **twice** is right 897 times. The first pass is the ranking
  above; the second keeps, for every member, the candidate whose URL shape most
  members agreed on. That recovers `developer.apple.com` to 100 of 108 and
  keeps the two pages the ranking repaired.

No new value is needed for the second pass: the members of one list agree on
the shape of their links, and that agreement says which link of a card is the
post.

In [6]:
def first_link(node):
    for n in node.walk():
        if n.link:
            return n.link
    return ""


rows = []
for p in pages:
    g = target_group(p)
    rows.append({
        "page": p.name,
        "members": len(g.children),
        "first link": sum(1 for l in [first_link(m) for m in g.children]
                          if p.is_post(l)),
        "one pass": sum(1 for l in [tree.representative(m, g.children)
                                    for m in g.children] if p.is_post(l)),
        "two passes": sum(1 for l in tree.group_links(g.children)
                          if p.is_post(l)),
    })
found = pd.DataFrame(rows).set_index("page")
found.loc["TOTAL"] = found.sum()
found

,members,first link,one pass,two passes
page,,,,
anthropic.com-news,10,10,10,10
aws.amazon.com-jp-blogs-news,11,10,10,10
bbc.com,15,3,12,12
blog.google,5,5,5,5
claude.com-blog,15,15,15,15
cursor.com-blog,12,12,12,12
daily.bandcamp.com-album-of-the-day,30,30,30,30
daily.bandcamp.com-features,30,30,30,30
deepmind.google-blog,12,12,12,12


### 3. A junk group looks exactly like a list of posts

A list of tags inside a card has one link per member, one shape and one
signature, just like a list of posts. What separates them is only what the
members hold: the text of a tag is a few characters, and a tag carries no date
and no image.

So no single value decides, which is why the rule below is a weighted score
rather than a filter.

In [7]:
rows = []
for p in pages:
    target = id(target_group(p))
    for g in p.groups():
        f = dict(features(g.children))
        f["page"] = p.name
        f["target"] = id(g) == target
        rows.append(f)
groups = pd.DataFrame(rows)

shown = ["n", "one_link_rate", "tpl_rate", "sig_rate", "med_text",
         "date_rate", "img_rate", "head_rate"]
groups.groupby("target")[shown].median()

,n,one_link_rate,tpl_rate,sig_rate,med_text,date_rate,img_rate,head_rate
target,,,,,,,,
False,2.0,0.5,0.6,0.75,37.0,0.0,0.0,0.0
True,15.0,1.0,1.0,1.00,64.0,1.0,1.0,0.0


In [8]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, column in zip(axes.flat, shown):
    data = [groups.loc[~groups.target, column], groups.loc[groups.target, column]]
    ax.boxplot(data, tick_labels=["other", "the list"], showfliers=False)
    ax.set_title(column)
fig.suptitle("What separates the list of posts from the other groups of a page")
fig.tight_layout()

## The score

`tree.score` combines four parts. The weights below were chosen by the grid
search further down.

```
score = 0.5 * content + 0.2 * shape + 0.2 * one_link_rate + 0.1 * size
```

- **content** is what the members hold: the median longest text, capped at 40
  characters, plus the share of members carrying a date, an image or a heading.
- **shape** is how much the members look alike: the share agreeing on the URL
  template, and the share agreeing on the signature.
- **one_link_rate** is the share of members holding exactly one candidate link
  of the shape the group agreed on.
- **size** grows with the number of members and is capped at eight.

The threshold is **relative to the best group of the same page**, because the
scores of two pages are not comparable: a page whose cards carry no date scores
lower everywhere.

## Two groups, one inside the other

A featured card at the top of a list is a group of its own, and the list sits
inside it. Both pass the threshold, so they have to be resolved, and how is a
real decision rather than a detail:

- **split** takes out of a group the members that hold another group already
  taken, so the featured card and the list beside it are offered as two
  groups. What is left of a group that lost members is scored again.
- **none** offers both as they are, so the posts of the list are shown twice.
- **first** keeps the one that scores higher. It is the cheapest for the user
  and it loses the whole list of `cursor.com` to a two-member featured pair.
- **larger** keeps the one with more members.

The last column is the share of the offered links that appear in more than one
of the selected groups, which is what the user would see twice.

In [9]:
def measure(alpha, score_fn=tree.score, nesting="split"):
    tp = fp = fn = taken = seen = twice = 0
    for p in pages:
        chosen = tree.select(p, score_fn=score_fn, alpha=alpha,
                             nesting=nesting, features_fn=features)
        taken += len(chosen)
        counts = {}
        for members in chosen:
            for link in tree.group_links(members):
                if link:
                    key = tree.normalize(link)
                    counts[key] = counts.get(key, 0) + 1
        seen += len(counts)
        twice += sum(1 for v in counts.values() if v > 1)
        predicted = set(counts)
        tp += len(predicted & p.posts)
        fp += len(predicted - p.posts)
        fn += len(p.posts - predicted)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return {"alpha": alpha, "groups": taken,
            "groups per page": round(taken / len(pages), 1),
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 3), "recall": round(recall, 3),
            "f1": round(2 * precision * recall / max(precision + recall, 1e-9), 3),
            "shown twice": round(twice / max(seen, 1), 3)}


rows = []
for nesting in ["split", "none", "first", "larger"]:
    for alpha in [0.9, 0.85, 0.8, 0.75]:
        rows.append({"nesting": nesting, **measure(alpha, nesting=nesting)})
pd.DataFrame(rows).set_index(["nesting", "alpha"])[
    ["groups per page", "precision", "recall", "f1", "shown twice"]]

groups per page  precision  recall     f1  shown twice
nesting alpha                                                        
split   0.90               2.0      0.984   0.896  0.938        0.037
        0.85               2.7      0.968   0.918  0.942        0.053
        0.80               3.4      0.971   0.969  0.970        0.058
        0.75               4.6      0.952   0.979  0.965        0.068
none    0.90               2.0      0.984   0.897  0.938        0.039
        0.85               2.9      0.960   0.919  0.939        0.057
        0.80               3.9      0.959   0.972  0.965        0.064
        0.75               5.1      0.939   0.982  0.960        0.088
first   0.90               1.8      0.984   0.886  0.933        0.026
        0.85               2.1      0.969   0.903  0.935        0.032
        0.80               2.5      0.966   0.924  0.945        0.031
        0.75               2.9      0.949   0.931  0.940        0.033
larger  0.90               2.0      0.985   0.896  0.938        0.037
        0.85               2.2      0.962   0.897  0.928        0.048
        0.80               2.9      0.965   0.948  0.956        0.053
        0.75               2.2      0.947   0.874  0.909        0.060

`split` is the best of the four at every threshold, and it is also the one that
matches what the page means: the hero post and the list beside it are two
different things to offer, not one of them instead of the other. It is the
default of `tree.select` and what the rest of the notebook uses.

In [10]:
sweep = pd.DataFrame([measure(a) for a in
                      [1.0, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65]])
sweep.set_index("alpha")[["groups per page", "tp", "fp", "fn",
                          "precision", "recall", "f1"]]

,groups per page,tp,fp,fn,precision,recall,f1
alpha,,,,,,,
1.00,1.0,879,9,191,0.990,0.821,0.898
0.95,1.2,904,9,166,0.990,0.845,0.912
0.90,2.0,959,16,111,0.984,0.896,0.938
0.85,2.7,982,32,88,0.968,0.918,0.942
0.80,3.4,1037,31,33,0.971,0.969,0.970
0.75,4.6,1047,53,23,0.952,0.979,0.965
0.70,5.7,1057,91,13,0.921,0.988,0.953
0.65,6.8,1055,122,15,0.896,0.986,0.939


In [11]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(sweep.alpha, sweep.precision, marker="o", label="precision")
left.plot(sweep.alpha, sweep.recall, marker="o", label="recall")
left.plot(sweep.alpha, sweep.f1, marker="o", label="F1")
left.invert_xaxis()
left.set_xlabel("alpha (share of the best score a group must reach)")
left.legend()
left.grid(alpha=0.3)
right.plot(sweep.alpha, sweep["groups per page"], marker="o", color="tab:red")
right.invert_xaxis()
right.set_xlabel("alpha")
right.set_ylabel("groups offered per page")
right.grid(alpha=0.3)
fig.suptitle("Taking more groups per page raises recall and costs the user ticks")
fig.tight_layout()

In [12]:
rows = []
for p in pages:
    chosen = tree.select(p, alpha=0.8, features_fn=features)
    predicted = tree.predicted_posts(chosen)
    rows.append({"page": p.name, "groups": len(chosen),
                 "posts": len(p.posts),
                 "tp": len(predicted & p.posts),
                 "fp": len(predicted - p.posts),
                 "fn": len(p.posts - predicted)})
per_page = pd.DataFrame(rows).set_index("page")
per_page["precision"] = (per_page.tp / (per_page.tp + per_page.fp)).round(2)
per_page["recall"] = (per_page.tp / (per_page.tp + per_page.fn)).round(2)
per_page

,groups,posts,tp,fp,fn,precision,recall
page,,,,,,,
anthropic.com-news,3,13,13,0,0,1.00,1.00
aws.amazon.com-jp-blogs-news,1,10,10,1,0,0.91,1.00
bbc.com,19,103,91,0,12,1.00,0.88
blog.google,1,8,5,0,3,1.00,0.62
claude.com-blog,4,24,23,0,1,1.00,0.96
cursor.com-blog,10,28,24,0,4,1.00,0.86
daily.bandcamp.com-album-of-the-day,2,30,30,7,0,0.81,1.00
daily.bandcamp.com-features,2,30,30,8,0,0.79,1.00
deepmind.google-blog,3,25,25,0,0,1.00,1.00


In [13]:
order = per_page.sort_values(["fn", "fp"], ascending=False)
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(order.index, order.tp, label="correct", color="tab:green")
ax.barh(order.index, order.fp, left=order.tp, label="wrong", color="tab:red")
ax.barh(order.index, -order.fn, label="missed", color="tab:orange")
ax.set_xscale("symlog")
ax.axvline(0, color="black", linewidth=0.8)
ax.legend()
ax.set_title("Posts found, posts missed and links wrongly offered (alpha = 0.8)")
fig.tight_layout()

## Are the weights fitted to these 21 pages?

Partly. The grid below searches the four weights and the threshold together.
The good settings form a wide plateau rather than a single peak, which means
the result does not depend on one exact number, but the top row is still chosen
with all 21 pages in view.

The honest figure is the leave-one-page-out run underneath: for each page, the
weights are chosen on the other twenty and measured on that page alone.

In [14]:
def weighted(weights):
    content_w, shape_w, one_w, size_w = weights

    def score_fn(f):
        content = (0.4 * min(f["med_text"] / 40.0, 1.0)
                   + 0.3 * f["date_rate"]
                   + 0.2 * f["img_rate"]
                   + 0.1 * f["head_rate"])
        shape = 0.5 * f["tpl_rate"] + 0.5 * f["sig_rate"]
        size = min(f["n"] / 8.0, 1.0)
        return (content_w * content + shape_w * shape + one_w * f["one_link_rate"]
                + size_w * size * f["link_rate"] * f["uniq_rate"])

    return score_fn


grid = []
for content_w in [0.3, 0.4, 0.5, 0.6]:
    for shape_w in [0.1, 0.2, 0.3]:
        for one_w in [0.1, 0.2, 0.3]:
            size_w = round(1 - content_w - shape_w - one_w, 2)
            if 0.05 <= size_w <= 0.3:
                for alpha in [0.85, 0.8, 0.75]:
                    grid.append(((content_w, shape_w, one_w, size_w), alpha))

results = []
for weights, alpha in grid:
    r = measure(alpha, weighted(weights))
    r["weights"] = weights
    results.append(r)
search = pd.DataFrame(results).sort_values("f1", ascending=False)
search[["weights", "alpha", "groups per page", "precision", "recall", "f1"]].head(10)

,weights,alpha,groups per page,precision,recall,f1
55,"(0.5, 0.3, 0.1, 0.1)",0.80,3.7,0.971,0.974,0.972
59,"(0.6, 0.1, 0.1, 0.2)",0.75,3.9,0.961,0.981,0.971
52,"(0.5, 0.2, 0.2, 0.1)",0.80,3.4,0.971,0.969,0.970
61,"(0.6, 0.1, 0.2, 0.1)",0.80,3.5,0.971,0.967,0.969
65,"(0.6, 0.2, 0.1, 0.1)",0.75,4.1,0.955,0.981,0.968
46,"(0.5, 0.1, 0.3, 0.1)",0.80,3.5,0.969,0.966,0.968
62,"(0.6, 0.1, 0.2, 0.1)",0.75,4.7,0.955,0.982,0.968
47,"(0.5, 0.1, 0.3, 0.1)",0.75,4.5,0.955,0.978,0.966
53,"(0.5, 0.2, 0.2, 0.1)",0.75,4.6,0.952,0.979,0.965
56,"(0.5, 0.3, 0.1, 0.1)",0.75,4.2,0.945,0.981,0.963


In [15]:
def counts(page, score_fn, alpha):
    chosen = tree.select(page, score_fn=score_fn, alpha=alpha,
                         features_fn=features)
    predicted = tree.predicted_posts(chosen)
    return (len(predicted & page.posts), len(predicted - page.posts),
            len(page.posts - predicted), len(chosen))


tp = fp = fn = taken = 0
for held_out in pages:
    rest = [p for p in pages if p is not held_out]
    best = None
    for weights, alpha in grid:
        score_fn = weighted(weights)
        a = b = c = 0
        for p in rest:
            x, y, z, _ = counts(p, score_fn, alpha)
            a, b, c = a + x, b + y, c + z
        precision = a / max(a + b, 1)
        recall = a / max(a + c, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)
        if best is None or f1 > best[0]:
            best = (f1, weights, alpha)
    x, y, z, g = counts(held_out, weighted(best[1]), best[2])
    tp, fp, fn, taken = tp + x, fp + y, fn + z, taken + g

precision = tp / (tp + fp)
recall = tp / (tp + fn)
print(f"leave one page out: {taken} groups, "
      f"precision {precision:.3f}, recall {recall:.3f}, "
      f"F1 {2 * precision * recall / (precision + recall):.3f}")

leave one page out: 79 groups, precision 0.963, recall 0.977, F1 0.970


## Text unwrapping

The idea: neighbouring texts written in tags that carry no meaning of their own
(`p`, `span`, `div` and the like) are one text. A list where some cards name
the author in a span and others do not then stops producing two different
shapes for the same list.

Measured on the group of each page that covers the most posts, the share of
members agreeing on one signature rises from 0.915 to 0.982, and no page gets
worse. Most of the gain comes from dropping the number of children and the
presence of an image from the signature, which a card that is still split into
several nodes would otherwise never match.

In [16]:
def signature_of(node, fold, kids, images):
    tags = {t["tag"] for t in node.texts}
    if fold:
        tags = {"text" if t in tree.NEUTRAL else t for t in tags}
    out = [bool(node.link), tuple(sorted(tags))]
    if images:
        out.append(bool(node.images))
    if kids:
        out.append(len(node.children))
    return tuple(out)


def modal_share(values):
    seen = {}
    for v in values:
        seen[v] = seen.get(v, 0) + 1
    return max(seen.values()) / len(values)


variants = {"tag of the element": (0, 1, 1),
            "neutral tags merged": (1, 1, 1),
            "without child count": (1, 0, 1),
            "without the image flag": (1, 0, 0)}
rows = []
for p in pages:
    g = target_group(p)
    row = {"page": p.name, "members": len(g.children)}
    for name, (fold, kids, images) in variants.items():
        row[name] = round(modal_share(
            [signature_of(c, fold, kids, images) for c in g.children]), 2)
    rows.append(row)
shapes = pd.DataFrame(rows).set_index("page")
mean = {name: round((shapes[name] * shapes.members).sum() / shapes.members.sum(), 3)
        for name in variants}
shapes.loc["weighted mean"] = {"members": shapes.members.sum(), **mean}
shapes

,members,tag of the element,neutral tags merged,without child count,without the image flag
page,,,,,
anthropic.com-news,10,1.000,1.000,1.000,1.000
aws.amazon.com-jp-blogs-news,11,0.910,0.910,0.910,0.910
bbc.com,15,0.670,0.670,0.800,0.800
blog.google,5,0.800,0.800,0.800,1.000
claude.com-blog,15,1.000,1.000,1.000,1.000
cursor.com-blog,12,1.000,1.000,1.000,1.000
daily.bandcamp.com-album-of-the-day,30,1.000,1.000,1.000,1.000
daily.bandcamp.com-features,30,1.000,1.000,1.000,1.000
deepmind.google-blog,12,0.920,1.000,1.000,1.000


The rule is right, and on this corpus it does not change the final numbers: the
shape part carries only 0.2 of the score, and the pages whose signatures
improve were already being found through their content. It would matter on a
page whose cards carry no date, no image and short titles, where the shape is
all there is to compare. The simplified signature is kept because it is never
worse and is simpler.

Merging the texts themselves, rather than only the signature, was measured too
and costs about 0.005 of F1: a merged text is longer, so a list of tags also
looks more like a list of posts.

## A homepage is not a blog index

`bbc.com` is offered as twenty groups, far more than any other page. That is
not a failure. Every one of them holds posts only, none of them repeats a post
of another, and they read as the sections of the homepage: sport, world news,
health, travel, food, science.

In [17]:
bbc = {p.name: p for p in pages}["bbc.com"]
seen = set()
rows = []
for members in sorted(tree.select(bbc, alpha=0.8, features_fn=features),
                      key=len, reverse=True):
    links = [l for l in tree.group_links(members) if l]
    posts = [l for l in links if bbc.is_post(l)]
    fresh = [l for l in posts if tree.normalize(l) not in seen]
    seen.update(tree.normalize(l) for l in posts)
    longest = max((t["value"] for t in members[0].all_texts()), key=len, default="")
    rows.append({"members": len(members), "posts": len(posts),
                 "not offered before": len(fresh),
                 "first member": longest[:52]})
pd.DataFrame(rows)

,members,posts,not offered before,first member
0,12,12,12,PL Review: Man Utd Struggle & Brighton Sensati...
1,12,12,12,"Dave Ramsey: Iran, tariffs and affordability"
2,8,8,8,"Man United Fan - ""CARRICK GONE!"""
3,4,4,4,Workers in northern Sri Lanka accidentally unc...
4,4,4,4,One eyewitness said the scene was so chaotic t...
5,4,4,4,"CNN, MS NOW and Politico reporters' White Hous..."
6,4,4,4,"While Burnham praised the agreement, Lib Dem l..."
7,4,4,4,How to find out how much state pension you're ...
8,4,4,4,Cyber Correspondent Joe Tidy looks at some of ...
9,4,4,4,Real Madrid manager Jose Mourinho prints out s...


The ten posts it does not reach sit in sections of two to four members that
fall just under the threshold; `alpha = 0.75` picks them up at the cost of more
groups elsewhere.

## What is still wrong

At `alpha = 0.8`:

In [18]:
mistakes = per_page[(per_page.fp > 0) | (per_page.fn > 0)]
mistakes.sort_values(["fn", "fp"], ascending=False)

,groups,posts,tp,fp,fn,precision,recall
page,,,,,,,
bbc.com,19,103,91,0,12,1.00,0.88
developer.apple.com-news,1,108,100,7,8,0.93,0.93
cursor.com-blog,10,28,24,0,4,1.00,0.86
github.blog,5,25,21,0,4,1.00,0.84
blog.google,1,8,5,0,3,1.00,0.62
claude.com-blog,4,24,23,0,1,1.00,0.96
ycombinator.com-blog,4,10,9,0,1,1.00,0.90
daily.bandcamp.com-features,2,30,30,8,0,0.79,1.00
daily.bandcamp.com-album-of-the-day,2,30,30,7,0,0.81,1.00


- **`blog.google`** loses three posts because their titles are not text at all.
  The page is built from custom elements and the title sits in an attribute:
  `<uni-simple-article-card headline="AI for everyone in every language" ...>`.
  Reading the text-like attributes of an element that has no text of its own
  would fix those three, and the semantic tree would have to carry them first.
- **`qiita.com`** and **`ycombinator.com`** give extra links rather than
  missing posts: their tag lists reach 80 % of the best score of the page.
- **`cursor.com`** lists press coverage on other hosts, so those links share no
  shape with the rest of the list.